# Phase 1: 2D Rendering + Pan/Zoom/Slab (Rust Backend)

This notebook exercises stateful render behavior through plane navigation, camera movement, and slab overrides on the Rust daemon.

In [1]:
from __future__ import annotations

import os
import shutil
import sys
import tempfile
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'tests').exists():
            return candidate
    raise RuntimeError('could not locate repository root from notebook cwd')


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
if str(REPO_ROOT / 'tests') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'tests'))

import base64
from io import BytesIO

from PIL import Image

from lucida.client import LucidaClient
from conftest import create_render_omezarr
from rust_daemon import start_rust_daemon


In [2]:
tmp_dir = Path(tempfile.mkdtemp(prefix='lucida-phase1-render-nav-rust-'))
dataset_uri = create_render_omezarr(str(tmp_dir / 'render.zarr'))
daemon = start_rust_daemon(repo_root=REPO_ROOT, env=dict(os.environ))
client = LucidaClient(base_url=daemon.base_url, backend='rust')
print({'dataset_uri': dataset_uri, 'base_url': daemon.base_url})


{'dataset_uri': '/var/folders/hs/qw7ws1q52153c4c639t_p3600000gn/T/lucida-phase1-render-nav-rust-q6x4wz0c/render.zarr', 'base_url': 'http://127.0.0.1:58606'}


In [3]:
opened = client.open_dataset(dataset_uri)
created = client.create_view(dataset_id=opened.dataset_summary.dataset_id)
view_id = created.view_state.view_id

client.set_plane(view_id=view_id, plane='xz')
client.pan(view_id=view_id, dx_px=12, dy_px=-6)
client.zoom(view_id=view_id, factor=0.75)

current = client.get_view(view_id=view_id).view_state
assert current.view_2d is not None
assert current.view_2d.plane == 'xz'
assert current.state_version >= 3

{'view_id': view_id, 'state_version': current.state_version, 'state_hash': current.state_hash}


{'view_id': 'view_40650000295e472a',
 'state_version': 3,
 'state_hash': 'd1cc881851dcae5ce60c3e52e558df431df4b699d84db8d2bc45525b35eb28da'}

In [4]:
rendered = client.render_image(
    view_id=view_id,
    width_px=96,
    height_px=64,
    overrides_json_patch=[
        {
            'op': 'replace',
            'path': '/view_2d/slice/slab',
            'value': {'thickness_vox': 5, 'mode': 'mip'},
        }
    ],
)

assert rendered.status == 'ok'
assert rendered.view_id == view_id
assert rendered.state_version == current.state_version
assert rendered.state_hash
assert rendered.images[0].mime == 'image/png'

image = Image.open(BytesIO(base64.b64decode(rendered.images[0].bytes_base64))).convert('RGBA')
assert image.size == (96, 64)

{'response_state_hash': rendered.state_hash, 'warnings': [w.code for w in rendered.warnings]}


{'response_state_hash': 'a7b31bbc72052fc47326b5bda2f47c731481b57f103837d54493c7207ec8fffc',
 'warnings': []}

In [5]:
single = client.render_image(
    view_id=view_id,
    width_px=64,
    height_px=48,
    overrides_json_patch=[
        {
            'op': 'replace',
            'path': '/view_2d/slice/slab',
            'value': {'thickness_vox': 5, 'mode': 'single'},
        }
    ],
)
assert single.status == 'ok'
single_image = Image.open(BytesIO(base64.b64decode(single.images[0].bytes_base64))).convert('RGBA')
assert single_image.size == (64, 48)
{warning.code for warning in single.warnings}


set()

In [6]:
if 'client' in globals():
    client.close()
if 'daemon' in globals():
    daemon.stop()
if 'tmp_dir' in globals():
    shutil.rmtree(tmp_dir, ignore_errors=True)
